# Find a resized image in a SQLite index
Can you find the original of a resized picture without comparing its pixels to every image in a collection? We'll index two sample images, query with a resized copy, and check the index against exhaustive search. Run the cells in order. The images ship with ImageHash.

In [ ]:
from pathlib import Path
import sqlite3
import sys
from tempfile import TemporaryDirectory

from IPython.display import display
from PIL import Image

# Support kernels started in the repository root or the examples directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples/sqlite_search.py").is_file())
sys.path.insert(0, str(root / "examples"))
from sqlite_search import bands, create_index, fingerprint, search, store

photos = root / "tests" / "data"
original = photos / "peppers.png"
scratch = TemporaryDirectory()
query_image = Path(scratch.name) / "query.jpg"
with Image.open(original) as image:
    preview = image.convert("RGB")
    preview.thumbnail((160, 160))
    preview.save(query_image, quality=80)
display(preview)

## Build the index
A 64-bit perceptual hash summarizes coarse image structure. Resizing and JPEG compression change pixel values, yet often leave most hash bits unchanged. The distance is the number of differing bits. Two different images can have the same hash, so treat a close result as a candidate to inspect.

SQLite stores the path, the complete hash and four indexed 16-bit pieces called bands. Using the path as the key lets us refresh the same image. Here the database lives in a temporary directory; the command-line example can keep it between sessions.

In [ ]:
db_path = Path(scratch.name) / "photos.sqlite"
with sqlite3.connect(db_path) as db:
    create_index(db)
    for path in (original, photos / "imagehash.png"):
        store(db, path.name, fingerprint(path))
db.close()

query = fingerprint(query_image)
print("Query hash:", query)
print("Four bands:", [format(value, "04x") for value in bands(query)])

## Search by distance
Try 0, 4, and 64. At zero, only identical fingerprints qualify. At 64, every 64-bit fingerprint qualifies. A useful threshold depends on the edits you expect and how much unrelated material you can review.

In [ ]:
radius = 4
db = sqlite3.connect(db_path)
for distance, name in search(db, query, radius):
    print(f"{distance:2d} bits different: {name}")
    with Image.open(photos / name) as match:
        match.thumbnail((160, 160))
        display(match.copy())

## Why four bands can skip comparisons
If at most three bits differ, at least one of four bands must match exactly: changing all four would take at least four bits. At distances four through seven, at least one band must differ in at most one bit. For each band, SQLite can look up its original value and its 16 one-bit neighbors.

Sharing a band only qualifies an image for a full distance check. At radius eight, two differences per band become possible, so this implementation switches to a full scan. Repeated bands can also make small-radius searches less selective. The four indexes trade extra storage and indexing time for faster candidate lookup.

On this small collection we can afford to check every threshold against every fingerprint.

In [ ]:
rows = list(db.execute("SELECT path, digest FROM phash64"))
exhaustive = sorted((bin(int(query, 16) ^ int(h, 16)).count("1"), path) for path, h in rows)
for radius in range(65):
    assert search(db, query, radius) == [row for row in exhaustive if row[0] <= radius]
print("All 65 thresholds matched exhaustive search.")
print("Distance to each indexed image:", exhaustive)
db.close()
scratch.cleanup()